In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_DIR = "/content/drive/MyDrive/morocco-purchasing-power/"

In [3]:
monthly_df = pd.read_csv(DATA_DIR + "hcp_monthly_category.csv")
yearly_df = pd.read_csv(DATA_DIR + "hcp_yearly_category.csv")
city_df = pd.read_csv(DATA_DIR + "hcp_city_indices.csv")

print(f"monthly: {monthly_df.shape}")
print(f"yearly: {yearly_df.shape}")
print(f"city: {city_df.shape}")

monthly: (1605, 6)
yearly: (1783, 9)
city: (2260, 9)


In [4]:
monthly_df.head()

,category,index_prev_period,index_curr_period,change_pct,month,year
0,Produits alimentaires,129.1,128.0,-0.8,Juin,2026
1,01 - Produits alimentaires et boissons non alc...,128.2,127.2,-0.8,Juin,2026
2,02 - Boissons alcoolisées et tabac,150.5,150.5,0.0,Juin,2026
3,Produits non alimentaires,114.8,114.6,-0.2,Juin,2026
4,03 - Articles d'habillements et chaussures,118.0,117.9,-0.1,Juin,2026


In [5]:
monthly_df.dtypes

,0
category,object
index_prev_period,float64
index_curr_period,float64
change_pct,float64
month,object
year,int64


In [6]:
monthly_df.isna().sum()

,0
category,31
index_prev_period,23
index_curr_period,23
change_pct,23
month,0
year,0


In [7]:
print(monthly_df.head())
print(monthly_df.dtypes)
print(monthly_df.isna().sum())

                                            category  index_prev_period  \
0                              Produits alimentaires              129.1   
1  01 - Produits alimentaires et boissons non alc...              128.2   
2                 02 - Boissons alcoolisées et tabac              150.5   
3                          Produits non alimentaires              114.8   
4         03 - Articles d'habillements et chaussures              118.0   

   index_curr_period  change_pct month  year  
0              128.0        -0.8  Juin  2026  
1              127.2        -0.8  Juin  2026  
2              150.5         0.0  Juin  2026  
3              114.6        -0.2  Juin  2026  
4              117.9        -0.1  Juin  2026  
category              object
index_prev_period    float64
index_curr_period    float64
change_pct           float64
month                 object
year                   int64
dtype: object
category             31
index_prev_period    23
index_curr_period    23
change_

In [8]:
fully_empty_mask = monthly_df["category"].isna() & monthly_df[["index_prev_period","index_curr_period","change_pct"]].isna().all(axis=1)
print(f"Dropping {fully_empty_mask.sum()} fully-empty junk rows")

Dropping 16 fully-empty junk rows


In [9]:
monthly_clean = monthly_df[~fully_empty_mask].copy()
print(f"Rows remaining: {len(monthly_clean)}")

Rows remaining: 1589


In [10]:
reference = monthly_clean[(monthly_clean["year"]==2010) & (monthly_clean["month"]=="février")]
category_template = reference["category"].tolist()
category_template

['Produits alimentaires',
 '01 - Produits alimentaires et boissons non alcoolisées',
 '02 - Boissons alcoolisées et tabac',
 'Produits non alimentaires',
 "03 - Articles d'habillements et chaussures",
 '04 - Logements, eau, électricité et autres combustibles',
 '05 - Meubles, articles de ménages et entretien courant du foyer',
 '06 - Santé',
 '07 - Transport',
 '08 - Communication',
 '09 - Loisirs et culture',
 '10 - Enseignement',
 '11 - Restaurants et hôtels',
 '12 - Biens et services divers',
 'Ensemble']

In [11]:
print(len(category_template))

15


In [12]:
still_missing = monthly_clean[monthly_clean["category"].isna()].groupby(["year","month"]).size()
still_missing

,,0
year,month,
2010,mars,15


In [13]:
mars_2010_idx = monthly_clean[(monthly_clean["year"]==2010) & (monthly_clean["month"]=="mars")].index
monthly_clean.loc[mars_2010_idx, "category"] = category_template

print(monthly_clean[(monthly_clean["year"]==2010) & (monthly_clean["month"]=="mars")])

                                               category  index_prev_period  \
1558                              Produits alimentaires              112.3   
1559  01 - Produits alimentaires et boissons non alc...              112.5   
1560                 02 - Boissons alcoolisées et tabac              108.3   
1561                          Produits non alimentaires              104.1   
1562         03 - Articles d'habillements et chaussures              104.5   
1563  04 - Logements, eau, électricité et autres com...              104.2   
1564  05 - Meubles, articles de ménages et entretien...              106.0   
1565                                         06 - Santé              102.5   
1566                                     07 - Transport              103.5   
1567                                 08 - Communication               90.9   
1568                            09 - Loisirs et culture               97.8   
1569                                  10 - Enseignement         

In [14]:
print("Remaining NaN categories:", monthly_clean["category"].isna().sum())
print("Total rows:", len(monthly_clean))

Remaining NaN categories: 0
Total rows: 1589


In [15]:
monthly_clean

,category,index_prev_period,index_curr_period,change_pct,month,year
0,Produits alimentaires,129.1,128.0,-0.8,Juin,2026
1,01 - Produits alimentaires et boissons non alc...,128.2,127.2,-0.8,Juin,2026
2,02 - Boissons alcoolisées et tabac,150.5,150.5,0.0,Juin,2026
3,Produits non alimentaires,114.8,114.6,-0.2,Juin,2026
4,03 - Articles d'habillements et chaussures,118.0,117.9,-0.1,Juin,2026
...,...,...,...,...,...,...
1600,09 - Loisirs et culture,98.1,98.0,-0.1,novembre,2009
1601,10 - Enseignement,109.1,113.3,3.8,novembre,2009
1602,11 - Restaurants et hôtels,105.1,107.8,2.6,novembre,2009
1603,12 - Biens et services divers,103.8,105.8,1.9,novembre,2009


In [16]:
garbled_mask = monthly_clean[["index_prev_period","index_curr_period","change_pct"]].isna().all(axis=1)
garbled_mask

,0
0,False
1,False
2,False
3,False
4,False
...,...
1600,False
1601,False
1602,False
1603,False


In [17]:
print(f"Dropping {garbled_mask.sum()} unrecoverable garbled rows: {monthly_clean[garbled_mask][['month','year']].values.tolist()}")
monthly_final = monthly_clean[~garbled_mask].copy()

Dropping 7 unrecoverable garbled rows: [['Juin', 2013], ['Février', 2013], ['Novembre', 2012], ['juin', 2011], ['mai', 2011], ['septembre', 2010], ['Juin', 2010]]


In [18]:
month_map = {
    "janvier": "Janvier", "février": "Février", "mars": "Mars", "avril": "Avril",
    "mai": "Mai", "juin": "Juin", "juillet": "Juillet", "août": "Août",
    "septembre": "Septembre", "octobre": "Octobre", "novembre": "Novembre", "décembre": "Décembre",
}
monthly_final["month"] = monthly_final["month"].str.lower().map(month_map)
print(monthly_final["month"].unique())

['Juin' 'Mai' 'Mars' 'Février' 'Novembre' 'Septembre' 'Juillet' 'Décembre']


In [19]:
print("Final shape:", monthly_final.shape)
print(monthly_final.isna().sum())

Final shape: (1582, 6)
category             0
index_prev_period    0
index_curr_period    0
change_pct           0
month                0
year                 0
dtype: int64


In [20]:
yearly_clean = yearly_df[yearly_df["category"].notna()].copy()
city_clean = city_df[city_df["city"].notna()].copy()
print("yearly:", yearly_clean.shape, "city:", city_clean.shape)

yearly: (1765, 9) city: (2242, 9)


In [21]:
value_cols = ["index_ref_period","index_curr_period","change_pct","cum_index_prior","cum_index_curr","cum_change_pct"]

yearly_garbled = yearly_clean[value_cols].isna().all(axis=1)
print(f"Dropping {yearly_garbled.sum()} garbled yearly rows")
yearly_clean = yearly_clean[~yearly_garbled].copy()

Dropping 8 garbled yearly rows


In [22]:
city_garbled = city_clean[value_cols].isna().all(axis=1)
print(f"Dropping {city_garbled.sum()} garbled city rows")
city_clean = city_clean[~city_garbled].copy()

Dropping 7 garbled city rows


In [23]:
yearly_clean.head()

,category,index_ref_period,index_curr_period,change_pct,cum_index_prior,cum_index_curr,cum_change_pct,month,year
0,Produits alimentaires,131.0,128.0,-2.3,131.5,130.2,-1.0,Juin,2026
1,01 - Produits alimentaires et boissons non alc...,130.5,127.2,-2.5,131.0,129.4,-1.2,Juin,2026
2,02 - Boissons alcoolisées et tabac,144.9,150.5,3.9,144.5,149.9,3.7,Juin,2026
3,Produits non alimentaires,112.0,114.6,2.3,112.1,113.8,1.5,Juin,2026
4,03 - Articles d'habillements et chaussures,117.0,117.9,0.8,116.8,117.9,0.9,Juin,2026


In [24]:
import re
canonical_map = {
    "01": "01 - Produits alimentaires et boissons non alcoolisées", "02": "02 - Boissons alcoolisées et tabac",
    "03": "03 - Articles d'habillement et chaussures", "04": "04 - Logement, eau, électricité et autres combustibles",
    "05": "05 - Meubles, articles de ménage et entretien courant du foyer", "06": "06 - Santé",
    "07": "07 - Transport", "08": "08 - Communication", "09": "09 - Loisirs et culture",
    "10": "10 - Enseignement", "11": "11 - Restaurants et hôtels", "12": "12 - Biens et services divers",
}

In [25]:
def normalize_category(cat):
    m = re.match(r"^(\d{2})\s*[-–]", cat)
    return canonical_map[m.group(1)] if m else cat

In [26]:
yearly_clean["category"] = yearly_clean["category"].apply(normalize_category)
print(yearly_clean["category"].nunique(), "unique categories")

15 unique categories


In [27]:
city_name_map = {
    "Al-hoceima": "Al-Hoceima", "Beni-Mellal": "Béni-Mellal", "Edakhla": "Dakhla",
}
city_clean["city"] = city_clean["city"].replace(city_name_map)
print(sorted(city_clean["city"].unique()))

['Agadir', 'Al-Hoceima', 'Béni-Mellal', 'Casablanca', 'Dakhla', 'Ensemble', 'Errachidia', 'Fès', 'Guelmim', 'Kénitra', 'Laâyoune', 'Marrakech', 'Meknès', 'Oujda', 'Rabat', 'Safi', 'Settat', 'Tanger', 'Tétouan']


In [28]:
city_clean.head()

,city,index_ref_period,index_curr_period,change_pct,cum_index_prior,cum_index_curr,cum_change_pct,month,year
0,Agadir,119.8,119.4,-0.3,118.8,119.7,0.8,Juin,2026
1,Casablanca,119.2,118.9,-0.3,118.5,119.1,0.5,Juin,2026
2,Fès,122.3,121.3,-0.8,122.6,122.4,-0.2,Juin,2026
3,Kénitra,121.7,119.6,-1.7,120.9,120.8,-0.1,Juin,2026
4,Marrakech,121.1,121.1,0.0,121.3,121.2,-0.1,Juin,2026


In [29]:
month_map = {
    "janvier":"Janvier","février":"Février","mars":"Mars","avril":"Avril","mai":"Mai","juin":"Juin",
    "juillet":"Juillet","août":"Août","septembre":"Septembre","octobre":"Octobre","novembre":"Novembre","décembre":"Décembre",
}
yearly_clean["month"] = yearly_clean["month"].str.lower().map(month_map)
city_clean["month"] = city_clean["month"].str.lower().map(month_map)
print(yearly_clean["month"].unique())

['Juin' 'Mai' 'Mars' 'Février' 'Janvier' 'Novembre' 'Septembre' 'Juillet'
 'Décembre']


In [30]:
print(city_clean["month"].unique())

['Juin' 'Mai' 'Mars' 'Février' 'Janvier' 'Novembre' 'Septembre' 'Juillet'
 'Décembre']


In [31]:
print("=== YEARLY ===")
print("shape:", yearly_clean.shape, "NaN:", yearly_clean.isna().sum().sum())
print("=== CITY ===")
print("shape:", city_clean.shape, "NaN:", city_clean.isna().sum().sum())

=== YEARLY ===
shape: (1757, 9) NaN: 0
=== CITY ===
shape: (2235, 9) NaN: 0


In [36]:
monthly_final.to_csv(DATA_DIR + "hcp_monthly_category_clean.csv", index=False)
yearly_clean.to_csv(DATA_DIR + "hcp_yearly_category_clean.csv", index=False)
city_clean.to_csv(DATA_DIR + "hcp_city_indices_clean.csv", index=False)
print("Saved all three clean datasets.")

Saved all three clean datasets.
